# GPPO 单种子机制验证（Colab 实时进度版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Battleplus/GPPO/blob/8.8-GPPO%E6%97%A0%E5%81%8F%E5%A5%BD/colab/GPPO_Mechanism_Validation_Colab.ipynb)

## Goal

本 Notebook 验证论文机制链条，而不是宣称论文级数值复现：

1. 异构任务图、动作掩码和 Eq.(1)–(5) 的公式级单元测试；
2. `GPPO/PPO × event/none` 的图结构与动态同步对照；
3. Adaptive、NoGate、SingleHead 消融；
4. 固定 test100 上与 Random、Greedy 比较；
5. 同一 GPPO checkpoint 的 Event/Full 重放及通信量比较；
6. 旧环境中一次确定性的领导机故障、重选与任务重分配探针。

默认复用 `/content/drive/MyDrive/GPPO_one_click/quick_seed1_100`，从 100 轮续训到 300 轮。找不到旧结果时从零训练。

> 结论边界：T5-10-48、训练 seed=1。test100 的实例级区间不能替代五个训练种子的置信区间；本结果只用于判断机制实现是否值得进入正式统计复现。


## Setup

请在 Colab 菜单选择 **运行时 → 更改运行时类型 → T4 GPU（或更高）**，然后按顺序运行全部单元格。训练结果会持续写入 Google Drive，断线后重新运行可续训。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Battleplus/GPPO.git"
BRANCH = "8.8-GPPO无偏好"
REPO = Path("/content/GPPO")

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "tqdm"], check=True)

import torch
print("repo:", REPO)
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 可调整参数。若只想复刻上次快速实验，把 TARGET_ITERATIONS 改为 100。
TARGET_ITERATIONS = 300
TRAIN_SEED = 1
TRAIN_JOBS = 2          # Colab T4/L4 建议 2；内存足够时可改 3–4
ROLLOUT_STEPS = 512
BATCH_SIZE = 512
UPDATE_EPOCHS = 4
VALIDATION_INTERVAL = 50
VALIDATION_INSTANCES = 20  # 与上次 100 轮断点保持一致；最终判断使用固定 test100
TEST_INSTANCES = 100
DISTURBANCE_INSTANCES = 20
PROGRESS_REFRESH_SECONDS = 5

DRIVE_ROOT = Path("/content/drive/MyDrive/GPPO_mechanism_validation")
RUN_ROOT = DRIVE_ROOT / f"mechanism_seed{TRAIN_SEED}_{TARGET_ITERATIONS}"
OLD_ROOT = Path("/content/drive/MyDrive/GPPO_one_click/quick_seed1_100")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

METHODS = [
    "literal:event",
    "ppo_mlp:none",
    "ppo_mlp:event",
    "literal:none",
    "literal_no_gate:event",
    "literal_single_head:event",
]

print("输出目录:", RUN_ROOT)
print("旧 100 轮目录存在:", OLD_ROOT.exists())
print("目标轮数:", TARGET_ITERATIONS, "并行训练数:", TRAIN_JOBS)


## Steps

### 1. 公式与环境机制测试

这一步先验证异构图、前序约束、动作掩码、固定事件带、Eq.(1)–(5)、RReLU、gate 非常数及有效梯度。测试失败会立即停止，不继续消耗训练时间。


In [ ]:
import json, time

formula_log = RUN_ROOT / "formula_tests.log"
formula_marker = RUN_ROOT / "formula_tests_passed.json"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

tests = [
    "tests/test_paper_faithful.py",
    "tests/test_paper_faithful_baselines.py",
    "tests/test_paper_faithful_resume.py",
    "tests/test_paper_faithful_cuda.py",
    "tests/test_report_paper_faithful_quick.py",
]
cmd = [sys.executable, "-m", "pytest", *tests, "-q"]
with formula_log.open("w", encoding="utf-8") as stream:
    result = subprocess.run(cmd, cwd=REPO, stdout=stream, stderr=subprocess.STDOUT)
print(formula_log.read_text(encoding="utf-8")[-5000:])
if result.returncode != 0:
    raise RuntimeError("公式/机制测试未通过；已停止训练，请查看 formula_tests.log")
formula_marker.write_text(json.dumps({"passed": True, "tests": tests, "timestamp": time.time()}, indent=2), encoding="utf-8")
print("✅ 公式与机制测试通过")


### 2. 导入旧断点并训练六个模型

若新目录尚不存在且检测到上次的 `quick_seed1_100`，会完整复制其 checkpoint、候选快照和历史，再由训练器从最近断点续训。训练日志分别保存在各模型目录，Colab 断线不会丢失已落盘的进度。


In [ ]:
import json

if OLD_ROOT.exists() and not (RUN_ROOT / "T5-10-48").exists():
    print("复制上次 100 轮结果用于续训……")
    shutil.copytree(OLD_ROOT, RUN_ROOT, dirs_exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "run_paper_faithful_formal.py",
    "--scale", "T5-10-48",
    "--seed", str(TRAIN_SEED),
    "--methods", *METHODS,
    "--iterations", str(TARGET_ITERATIONS),
    "--rollout-steps", str(ROLLOUT_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--update-epochs", str(UPDATE_EPOCHS),
    "--validation-interval", str(VALIDATION_INTERVAL),
    "--validation-instances", str(VALIDATION_INSTANCES),
    "--device", "auto",
    "--rrelu-mode", "expected",
    "--gate-scope", "task_message",
    "--gate-activation", "sigmoid",
    "--jobs", str(TRAIN_JOBS),
    "--output-root", str(RUN_ROOT),
    "--rerun-completed",
]
print(" ".join(cmd))

# 训练进程在后台运行；本单元格持续读取 Drive 上的 history，实时刷新进度。
import time
from datetime import timedelta
import pandas as pd
from IPython.display import HTML, display
from tqdm.notebook import tqdm

MODEL_DIRECTORIES = {
    "GPPO-event": "literal_event_seed1",
    "PPO-none": "ppo_mlp_none_seed1",
    "PPO-event": "ppo_mlp_event_seed1",
    "GPPO-none": "literal_none_seed1",
    "NoGate-event": "literal_no_gate_event_seed1",
    "SingleHead-event": "literal_single_head_event_seed1",
}
scale_root = RUN_ROOT / "T5-10-48"

def read_training_progress(model_directory):
    history_path = scale_root / model_directory / "training_history.json"
    if not history_path.exists():
        return {"iteration": 0, "status": "等待启动"}
    try:
        history = json.loads(history_path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {"iteration": 0, "status": "正在写入"}
    if not history:
        return {"iteration": 0, "status": "已启动"}
    row = dict(history[-1])
    iteration = min(TARGET_ITERATIONS, int(float(row.get("iteration", 0))))
    row["iteration"] = iteration
    row["status"] = "训练完成" if iteration >= TARGET_ITERATIONS else ("断点续训中" if iteration >= 100 else "训练中")
    return row

def hardware_status():
    try:
        value = subprocess.check_output([
            "nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip().splitlines()[0]
        utilization, used, total = [part.strip() for part in value.split(",")]
        return f"GPU {utilization}% | VRAM {used}/{total} MiB"
    except Exception:
        return "GPU unavailable / CPU mode"

training_log_path = RUN_ROOT / "colab_training_launcher.log"
training_log = training_log_path.open("a", encoding="utf-8")
process = subprocess.Popen(cmd, cwd=REPO, stdout=training_log, stderr=subprocess.STDOUT, text=True)

bars = {
    name: tqdm(total=TARGET_ITERATIONS, desc=f"{name:20s}", unit="iter", leave=True)
    for name in MODEL_DIRECTORIES
}
overall_bar = tqdm(total=TARGET_ITERATIONS * len(MODEL_DIRECTORIES), desc="总体训练进度", unit="iter", leave=True)
stage_display = display(HTML("<b>正在启动六模型训练……</b>"), display_id=True)
metrics_display = display(pd.DataFrame(), display_id=True)
started_at = time.monotonic()
initial_total = None

try:
    while process.poll() is None:
        rows, total_progress, completed_models = [], 0, 0
        for name, directory in MODEL_DIRECTORIES.items():
            row = read_training_progress(directory)
            iteration = int(row.get("iteration", 0))
            total_progress += iteration
            completed_models += int(iteration >= TARGET_ITERATIONS)
            bars[name].n = iteration
            bars[name].set_postfix_str(row.get("status", "等待启动"), refresh=True)
            rows.append({
                "模型": name,
                "状态": row.get("status", "等待启动"),
                "进度": f"{iteration}/{TARGET_ITERATIONS}",
                "百分比": f"{100.0 * iteration / TARGET_ITERATIONS:.1f}%",
                "reward": round(float(row["reward"]), 4) if "reward" in row else None,
                "makespan": round(float(row["realized_makespan"]), 4) if "realized_makespan" in row else None,
                "loss": round(float(row["loss"]), 4) if "loss" in row else None,
            })
        if initial_total is None:
            initial_total = total_progress
        overall_bar.n = total_progress
        overall_bar.refresh()
        elapsed_seconds = max(1.0, time.monotonic() - started_at)
        progressed = max(0, total_progress - initial_total)
        rate = progressed / elapsed_seconds
        remaining = max(0, TARGET_ITERATIONS * len(MODEL_DIRECTORIES) - total_progress)
        eta = str(timedelta(seconds=int(remaining / rate))) if rate > 1e-6 else "计算中"
        stage_display.update(HTML(
            f"<h4>当前阶段：六模型训练（已完成 {completed_models}/6）</h4>"
            f"<p>已用时间：{timedelta(seconds=int(elapsed_seconds))}　|　预计剩余：{eta}　|　"
            f"{hardware_status()}　|　并发任务：{TRAIN_JOBS}</p>"
            f"<p>训练日志：<code>{training_log_path}</code></p>"
        ))
        metrics_display.update(pd.DataFrame(rows))
        time.sleep(PROGRESS_REFRESH_SECONDS)
finally:
    training_log.close()
    for bar in bars.values():
        bar.close()
    overall_bar.close()

return_code = process.wait()
if return_code != 0:
    tail = training_log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]
    print("\n".join(tail))
    raise RuntimeError(f"六模型训练失败，退出码={return_code}；请查看 {training_log_path}")

stage_display.update(HTML("<h3 style='color:green'>六模型训练全部完成，开始 test100 与机制诊断。</h3>"))
print("✅ 六模型训练完成；完整日志：", training_log_path)


### 3. 固定 test100、基线、通信重放与 gate 诊断

六个模型使用同一固定测试实例库和同一事件带。Full 通信只重放 `GPPO-event` 的同一 checkpoint，不重新训练，因此质量差与通信差可归因于同步模式。


In [ ]:
# 六模型原生通信模式 test100
subprocess.run([
    sys.executable, "evaluate_paper_faithful_formal.py",
    "--root", str(RUN_ROOT), "--instances", str(TEST_INSTANCES),
    "--split", "test", "--jobs", "2",
], cwd=REPO, check=True)

gppo_dir = RUN_ROOT / "T5-10-48" / f"literal_event_seed{TRAIN_SEED}"
gppo_ckpt = gppo_dir / "checkpoint.pt"

# 同 checkpoint 的 Full/always 重放
subprocess.run([
    sys.executable, "evaluate_paper_faithful.py",
    "--checkpoint", str(gppo_ckpt), "--instances", str(TEST_INSTANCES),
    "--split", "test", "--sync-mode", "always",
    "--output", str(gppo_dir / "evaluations" / f"test_native_always_{TEST_INSTANCES}.json"),
], cwd=REPO, check=True)

# Random / Greedy
baseline_path = RUN_ROOT / f"baselines_test{TEST_INSTANCES}.json"
subprocess.run([
    sys.executable, "evaluate_paper_faithful_baselines.py",
    "--scale", "T5-10-48", "--instances", str(TEST_INSTANCES),
    "--policy-seed", str(TRAIN_SEED), "--output", str(baseline_path),
], cwd=REPO, check=True)

# Adaptive gate 分布与两条梯度路径
gate_path = gppo_dir / "gate_diagnostic.json"
subprocess.run([
    sys.executable, "diagnose_paper_faithful_gate.py",
    "--checkpoint", str(gppo_ckpt), "--split", "test", "--instances", "100",
    "--output", str(gate_path),
], cwd=REPO, check=True)
print("✅ test100、Random/Greedy、Event/Full 与 gate 诊断完成")


### 4. 旧环境领导机故障探针

这里只加入论文机制中已有的 UAV/leader failure，不加入后续扩展提出的 Gilbert–Elliott、随机网络时延、风场或任务取消。每个实例先保证四类能力各有一架非 leader 备份机，再在领导机执行任务后强制故障，记录：故障检测、leader 重选、运行中任务释放与再次完成。能力冗余条件可避免“唯一具备能力的无人机死亡后任务物理不可完成”被误判成同步代码错误。该探针是结构正确性证据，不属于主 test100 排名。


In [ ]:
%%writefile /content/GPPO/leader_failure_probe.py
from __future__ import annotations

import argparse, json, statistics, sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent / "src"))
from train_paper_faithful import tensors
from uav_assignment.paper_env import PaperAlignedUAVEnv, PaperEnvConfig
from uav_assignment.paper_faithful_models import PaperFaithfulActorCritic


def inject_leader_failure(env: PaperAlignedUAVEnv, sync_mode: str) -> dict:
    victim = env.leader_id
    if victim < 0 or not env.uavs[victim].alive:
        raise RuntimeError("no live leader to fail")
    was_busy = env.uavs[victim].busy_task
    records = [env._event_record(
        "uav_failure", "mechanism_probe", "uav", victim,
        before={"alive": True, "leader": True}, after={"alive": False, "leader": False},
        forced=True,
    )]
    env.uavs[victim].alive = False
    env.uavs[victim].health = 0.0
    records.append(env._event_record("leader_failure", "mechanism_probe", "uav", victim, True, False, forced=True))
    env.leader_id = -1
    if was_busy >= 0:
        task = env.tasks[was_busy]
        task.assigned_uav = -1
        task.start_time = -1.0
        task.expected_completion = -1.0
        task.processing_time = 0.0
        task.reallocation_attempts += 1
        env.uavs[victim].busy_task = -1
        env.uavs[victim].remaining_time = 0.0
        env.reallocated_tasks += 1
        records.append(env._event_record(
            "task_reallocated", "uav_failure", "task", was_busy,
            before={"assigned_uav": victim}, after={"assigned_uav": -1}, forced=True,
        ))
    elected = env._first_alive_uav(env.uavs)
    env.leader_id = elected
    if elected >= 0:
        env.leader_changes += 1
        records.append(env._event_record(
            "leader_elected", "leader_failure", "uav", elected,
            before=-1, after=elected, forced=True,
        ))
    updated = env._synchronize_belief(count_communication=True, full=sync_mode == "always", records=records)
    for record in records:
        record.triggered_sync = sync_mode == "event"
        record.details.setdefault("sync_mode", sync_mode)
        record.details.setdefault("sync_reason", "forced_leader_failure_probe")
        env.event_log.append(record.to_dict())
    return {"failed_uav": victim, "released_task": was_busy, "elected_uav": elected, "updated_uavs": updated}


def run_episode(model, model_cfg, seed: int, sync_mode: str, fail: bool) -> dict:
    env = PaperAlignedUAVEnv(PaperEnvConfig(
        max_uavs=int(model_cfg["max_uavs"]), max_tasks=int(model_cfg["max_tasks"]),
        active_uavs=5, initial_tasks=48, max_decisions=500,
        # 故障探针聚焦“重选/释放/恢复”链条，避免唯一能力无人机失效后
        # 任务在物理上不可完成，把不可行性误判成同步机制错误。
        capability_threshold=0.01,
        weather_probability=0.0, failure_probability=0.0,
        task_change_probability=0.0, communication_drop_probability=0.0,
        include_engineering_rewards=False, mission_deadline=0.0,
        stop_at_deadline=False, task_chain_length=4, seed=seed,
    ))
    env.reset(seed=seed)
    # 原旧环境会把 search 能力几乎全部集中在 leader；直接杀死 leader 会造成
    # 物理不可行，而非重分配机制失败。为每类任务指定一架非 leader 备份机。
    backups = [index for index, uav in enumerate(env.uavs) if uav.active and uav.alive and index != env.leader_id]
    if not backups:
        raise RuntimeError("leader failure probe needs at least one backup UAV")
    for task_type in range(4):
        backup = backups[task_type % len(backups)]
        env.uavs[backup].capabilities[task_type] = max(0.35, float(env.uavs[backup].capabilities[task_type]))
    env._synchronize_belief(count_communication=False, full=True)
    observation = env.observe()
    injected = None
    done = False
    while not done:
        # 等领导机真正承担任务后再故障，确保“释放—重分配—完成”链条被执行。
        if fail and injected is None and env.decision_count >= 10 and env.leader_id >= 0 and env.uavs[env.leader_id].busy_task >= 0:
            injected = inject_leader_failure(env, sync_mode)
            observation = env.observe()
        with torch.no_grad():
            action, _, _ = model.act(tensors(observation), deterministic=True)
        observation, _, done, _ = env.step(int(action.item()), sync_mode=sync_mode)
    metrics = env.metrics()
    event_types = [row["event_type"] for row in env.event_log]
    released = -1 if injected is None else int(injected["released_task"])
    recovered = bool(released >= 0 and env.tasks[released].completed and env.tasks[released].recovered_after_reallocation)
    return {
        "seed": seed, "sync_mode": sync_mode, "failure_injected": injected is not None,
        "injection": injected, "event_types": event_types,
        "leader_failure_seen": "leader_failure" in event_types,
        "leader_elected_seen": "leader_elected" in event_types,
        "task_reallocated_seen": "task_reallocated" in event_types,
        "released_task_recovered": recovered,
        **metrics,
    }


def mean(rows, key):
    return statistics.fmean(float(row[key]) for row in rows)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--instances", type=int, default=20)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    model_cfg = checkpoint["model_config"]
    model = PaperFaithfulActorCritic(**model_cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    seeds = [910000 + index for index in range(args.instances)]
    clean = [run_episode(model, model_cfg, seed, "event", False) for seed in seeds]
    event_failure = [run_episode(model, model_cfg, seed, "event", True) for seed in seeds]
    full_failure = [run_episode(model, model_cfg, seed, "always", True) for seed in seeds]
    payload = {
        "version": "deterministic-leader-failure-probe-v1",
        "scope": "old engineering environment; redundancy-conditioned; one forced leader failure after leader becomes busy",
        "instances": args.instances,
        "rows": {"clean_event": clean, "failure_event": event_failure, "failure_full": full_failure},
        "summary": {
            "clean_event_makespan": mean(clean, "makespan"),
            "failure_event_makespan": mean(event_failure, "makespan"),
            "failure_full_makespan": mean(full_failure, "makespan"),
            "failure_event_completion": mean(event_failure, "completion_rate"),
            "failure_full_completion": mean(full_failure, "completion_rate"),
            "event_failure_invalid_actions": mean(event_failure, "invalid_actions"),
            "event_failure_reallocation_success_rate": mean(event_failure, "reallocation_success_rate"),
            "failure_injection_coverage": mean(event_failure, "failure_injected"),
            "leader_election_coverage": mean(event_failure, "leader_elected_seen"),
            "task_reallocation_coverage": mean(event_failure, "task_reallocated_seen"),
            "released_task_recovery_coverage": mean(event_failure, "released_task_recovered"),
        },
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(payload["summary"], indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


In [ ]:
disturbance_path = RUN_ROOT / "leader_failure_probe.json"
subprocess.run([
    sys.executable, "leader_failure_probe.py",
    "--checkpoint", str(gppo_ckpt),
    "--instances", str(DISTURBANCE_INSTANCES),
    "--output", str(disturbance_path),
], cwd=REPO, check=True)
print("✅ 领导机故障机制探针完成")


## Checks

最后一个单元格执行证据审计、输出中文报告、生成图和 zip。核心机制结论和故障探针分开判定：故障探针失败不会被隐藏，也不会被误写成论文原始 test100 的失败。


In [ ]:
import hashlib, math, statistics
import matplotlib.pyplot as plt

LABELS = {
    "GPPO-event": "literal_event_seed1",
    "PPO-none": "ppo_mlp_none_seed1",
    "PPO-event": "ppo_mlp_event_seed1",
    "GPPO-none": "literal_none_seed1",
    "GPPO-NoGate-event": "literal_no_gate_event_seed1",
    "GPPO-SingleHead-event": "literal_single_head_event_seed1",
}

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def metric(payload, key, stat="mean"):
    return float(payload["summary"][key][stat])

def paired(left, right, key="realized_makespan"):
    a = {int(r["instance_seed"]): float(r[key]) for r in left["rows"]}
    b = {int(r["instance_seed"]): float(r[key]) for r in right["rows"]}
    if set(a) != set(b):
        raise ValueError("paired test banks differ")
    values = [a[s] - b[s] for s in sorted(a)]
    mean = statistics.fmean(values)
    sd = statistics.stdev(values) if len(values) > 1 else 0.0
    half = 1.984216951 * sd / math.sqrt(len(values)) if len(values) > 1 else 0.0
    return {"mean": mean, "median": statistics.median(values), "ci95_low": mean-half, "ci95_high": mean+half, "n": len(values)}

scale_dir = RUN_ROOT / "T5-10-48"
evaluations, checkpoints = {}, {}
protocol_errors = []
for name, directory in LABELS.items():
    model_dir = scale_dir / directory
    checkpoint_path = model_dir / "checkpoint.pt"
    evaluation_path = model_dir / "evaluations" / f"test_native_{TEST_INSTANCES}.json"
    if not checkpoint_path.exists() or not evaluation_path.exists():
        raise FileNotFoundError(f"missing final artifact for {name}")
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    checkpoints[name] = ckpt
    evaluations[name] = load_json(evaluation_path)
    training = ckpt.get("training", {})
    expected = {
        "iterations": TARGET_ITERATIONS, "rollout_steps": ROLLOUT_STEPS,
        "batch_size": BATCH_SIZE, "update_epochs": UPDATE_EPOCHS,
        "validation_interval": VALIDATION_INTERVAL,
        "validation_instances": VALIDATION_INSTANCES, "seed": TRAIN_SEED,
    }
    for key, value in expected.items():
        if int(training.get(key, -1)) != int(value):
            protocol_errors.append(f"{name}: {key}={training.get(key)} expected={value}")
    if len(ckpt.get("history", [])) != TARGET_ITERATIONS:
        protocol_errors.append(f"{name}: history={len(ckpt.get('history', []))} expected={TARGET_ITERATIONS}")

test_sets = [{int(r["instance_seed"]) for r in p["rows"]} for p in evaluations.values()]
tape_sets = [set(p.get("event_tape_hashes", [])) for p in evaluations.values()]
same_test_bank = all(len(s) == TEST_INSTANCES and s == test_sets[0] for s in test_sets)
same_event_tapes = bool(tape_sets[0]) and all(s == tape_sets[0] for s in tape_sets)

full = load_json(gppo_dir / "evaluations" / f"test_native_always_{TEST_INSTANCES}.json")
gppo = evaluations["GPPO-event"]
baseline_payload = load_json(baseline_path)
baselines = {r["policy"]: r for r in baseline_payload["results"] if r["scale"] == "T5-10-48" and int(r["policy_seed"]) == TRAIN_SEED}
random_ms = float(baselines["random"]["summary"]["realized_makespan"]["mean"])
greedy_ms = float(baselines["greedy"]["summary"]["realized_makespan"]["mean"])

comparisons = {
    "GPPO-event_vs_PPO-none": paired(gppo, evaluations["PPO-none"]),
    "GPPO-event_vs_PPO-event": paired(gppo, evaluations["PPO-event"]),
    "GPPO-none_vs_PPO-none": paired(evaluations["GPPO-none"], evaluations["PPO-none"]),
    "GPPO-event_vs_GPPO-none": paired(gppo, evaluations["GPPO-none"]),
    "Adaptive_vs_NoGate": paired(gppo, evaluations["GPPO-NoGate-event"]),
    "Adaptive_vs_SingleHead": paired(gppo, evaluations["GPPO-SingleHead-event"]),
}

gppo_ms = metric(gppo, "realized_makespan")
full_ms = metric(full, "realized_makespan")
event_bytes = metric(gppo, "communication_bytes")
full_bytes = metric(full, "communication_bytes")
quality_gap = 100.0 * (gppo_ms-full_ms) / max(abs(full_ms), 1e-12)
comm_reduction = 100.0 * (full_bytes-event_bytes) / max(full_bytes, 1e-12)
same_checkpoint = gppo.get("checkpoint_sha256") == full.get("checkpoint_sha256")
same_replay_tapes = set(gppo.get("event_tape_hashes", [])) == set(full.get("event_tape_hashes", []))

gate = load_json(gate_path)
disturbance = load_json(disturbance_path)
ds = disturbance["summary"]

checks = {
    "formula_and_mechanism_tests_passed": formula_marker.exists(),
    "training_protocol_exact": not protocol_errors,
    "same_fixed_test100": same_test_bank,
    "same_fixed_event_tapes": same_event_tapes,
    "gppo_event_better_than_ppo_none": comparisons["GPPO-event_vs_PPO-none"]["mean"] < 0,
    "graph_benefit_under_event": comparisons["GPPO-event_vs_PPO-event"]["mean"] < 0,
    "graph_benefit_under_none": comparisons["GPPO-none_vs_PPO-none"]["mean"] < 0,
    "adaptive_better_than_nogate": comparisons["Adaptive_vs_NoGate"]["mean"] < 0,
    "adaptive_better_than_singlehead": comparisons["Adaptive_vs_SingleHead"]["mean"] < 0,
    "gppo_better_than_random": gppo_ms < random_ms,
    "gppo_within_5_percent_of_greedy": gppo_ms <= 1.05 * greedy_ms,
    "event_full_same_checkpoint": same_checkpoint,
    "event_full_same_tapes": same_replay_tapes,
    "event_within_5_percent_of_full": abs(quality_gap) <= 5.0,
    "event_uses_fewer_bytes": event_bytes < full_bytes,
    "gate_not_saturated_constant": gate["gate"]["std"] > 1e-4 and gate["gate"]["fraction_gt_0_9"] < 0.99,
    "gate_has_policy_gradient": float(gate["gate_gradient_l2_policy_sensitivity"]) > 0,
    "gate_has_ppo_probe_gradient": float(gate["gate_gradient_l2_actual_ppo_probe"]) > 0,
}
disturbance_checks = {
    "failure_injected_all_instances": float(ds["failure_injection_coverage"]) == 1.0,
    "leader_elected_all_instances": float(ds["leader_election_coverage"]) == 1.0,
    "running_task_released_all_instances": float(ds["task_reallocation_coverage"]) == 1.0,
    "released_task_recovered_all_instances": float(ds["released_task_recovery_coverage"]) == 1.0,
    "failure_event_completion_at_least_95_percent": float(ds["failure_event_completion"]) >= 0.95,
}

core_pass = all(checks.values())
disturbance_pass = all(disturbance_checks.values())
if core_pass and disturbance_pass:
    decision = "机制验证通过；继续五种子正式统计复现"
elif core_pass:
    decision = "论文主机制方向通过，但旧环境故障恢复探针未完全通过；先修故障链再做扩展"
else:
    decision = "机制验证未通过；暂停 PCRL/世界模型接入，按失败检查项修正 GPPO"

result = {
    "version": "gppo-mechanism-validation-v2",
    "scope": f"T5-10-48, training seed={TRAIN_SEED}, {TARGET_ITERATIONS} iterations, fixed test{TEST_INSTANCES}",
    "not_claimed": "not a five-seed paper-level statistical reproduction",
    "decision": decision,
    "checks": checks,
    "disturbance_checks": disturbance_checks,
    "protocol_errors": protocol_errors,
    "methods": {name: {"return": p["summary"]["episode_return"], "makespan": p["summary"]["realized_makespan"], "completion": p["summary"]["completion_rate"], "communication_bytes": p["summary"]["communication_bytes"]} for name, p in evaluations.items()},
    "comparisons": comparisons,
    "baselines": {"random_makespan": random_ms, "greedy_makespan": greedy_ms},
    "communication": {"event_makespan": gppo_ms, "full_makespan": full_ms, "quality_gap_percent": quality_gap, "event_bytes": event_bytes, "full_bytes": full_bytes, "reduction_percent": comm_reduction},
    "gate": gate,
    "leader_failure_summary": ds,
}

json_path = RUN_ROOT / "MECHANISM_ACCEPTANCE.json"
json_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

lines = [
    "# GPPO 单种子机制验证报告", "",
    f"> 范围：{result['scope']}。仅用于机制方向筛查，不能替代五训练种子的论文级统计复现。", "",
    "## 结论", "", f"**{decision}**", "",
    "## 六模型 test100", "",
    "| 方法 | Return mean | Makespan mean | Makespan median | Completion | Comm bytes |",
    "|---|---:|---:|---:|---:|---:|",
]
for name, p in evaluations.items():
    lines.append(f"| {name} | {metric(p, 'episode_return'):.4f} | {metric(p, 'realized_makespan'):.4f} | {metric(p, 'realized_makespan', 'median'):.4f} | {metric(p, 'completion_rate'):.4f} | {metric(p, 'communication_bytes'):.1f} |")
lines += ["", "## 配对 test100 差值", "", "负 makespan 差值表示左侧方法更好；该区间是实例级区间，不是训练种子置信区间。", "", "| 对比 | Mean Δ | Median Δ | 95% CI |", "|---|---:|---:|---:|"]
for name, row in comparisons.items():
    lines.append(f"| {name} | {row['mean']:.4f} | {row['median']:.4f} | [{row['ci95_low']:.4f}, {row['ci95_high']:.4f}] |")
lines += [
    "", "## Random / Greedy", "", f"- Random makespan：{random_ms:.4f}", f"- Greedy makespan：{greedy_ms:.4f}",
    "", "## Event / Full 同 checkpoint 重放", "", f"- Event / Full makespan：{gppo_ms:.4f} / {full_ms:.4f}", f"- 质量差：{quality_gap:.2f}%", f"- Event / Full bytes：{event_bytes:.1f} / {full_bytes:.1f}", f"- 通信减少率：{comm_reduction:.2f}%",
    "", "## Adaptive gate 诊断", "", f"- gate mean/std：{gate['gate']['mean']:.6f} / {gate['gate']['std']:.6f}", f"- policy sensitivity gradient L2：{gate['gate_gradient_l2_policy_sensitivity']:.6g}", f"- PPO probe gradient L2：{gate['gate_gradient_l2_actual_ppo_probe']:.6g}",
    "", "## 旧环境领导机故障探针", "", f"- 故障/重选/任务释放/任务恢复覆盖率：{ds['failure_injection_coverage']:.2%} / {ds['leader_election_coverage']:.2%} / {ds['task_reallocation_coverage']:.2%} / {ds['released_task_recovery_coverage']:.2%}", f"- clean event / failure event / failure full makespan：{ds['clean_event_makespan']:.4f} / {ds['failure_event_makespan']:.4f} / {ds['failure_full_makespan']:.4f}", f"- failure-event completion：{ds['failure_event_completion']:.2%}",
    "", "## 验收检查", "",
]
for name, passed in checks.items():
    lines.append(f"- {'PASS' if passed else 'FAIL'} — {name}")
lines += ["", "### 故障探针", ""]
for name, passed in disturbance_checks.items():
    lines.append(f"- {'PASS' if passed else 'FAIL'} — {name}")
lines += [
    "", "## 结论边界", "",
    "- 已验证的是公式实现、机制消融和固定实例上的行为，不是原论文数值级复现。",
    "- 单个训练 seed 无法证明 GPPO 稳定优于 PPO/Greedy；正式结论仍需至少五个训练 seed。",
    "- 领导机故障属于旧工程环境诊断，未混入论文主 test100 排名。",
]
report_path = RUN_ROOT / "GPPO_MECHANISM_REPORT_ZH.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

# 两张简单图，便于在 Colab 直接看方向。
names = list(evaluations)
means = [metric(evaluations[n], "realized_makespan") for n in names]
plt.figure(figsize=(11, 4.5))
plt.bar(names, means)
plt.axhline(greedy_ms, color="tab:orange", linestyle="--", label="Greedy")
plt.ylabel("Realized makespan (lower is better)")
plt.xticks(rotation=25, ha="right")
plt.legend(); plt.tight_layout()
plt.savefig(RUN_ROOT / "mechanism_makespan.png", dpi=180)
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(["Clean-event", "Failure-event", "Failure-full"], [ds["clean_event_makespan"], ds["failure_event_makespan"], ds["failure_full_makespan"]])
plt.ylabel("Makespan")
plt.tight_layout()
plt.savefig(RUN_ROOT / "leader_failure_makespan.png", dpi=180)
plt.show()

archive_base = DRIVE_ROOT / f"GPPO_mechanism_seed{TRAIN_SEED}_{TARGET_ITERATIONS}"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
print("\n".join(lines[:35]))
print("\n报告:", report_path)
print("审计 JSON:", json_path)
print("压缩包:", archive)


## Next Steps

- 若所有主机制检查均为 PASS：进入四尺度 × 五训练种子的正式统计复现。
- 若 Adaptive 对 NoGate 或 SingleHead 为 FAIL：暂停“adaptive 有益”的表述，检查 gate 作用域、RReLU、softmax 域和优化方差。
- 若 GPPO 对 PPO/Random 为 FAIL：暂停 PCRL 与世界模型接入，先修 GPPO 基线。
- 若只有故障探针 FAIL：主论文机制与工程故障恢复分开处理，优先修复 leader 重选后的 belief 同步或任务释放链。
